In [2]:
#import json
from statistics import mean,median,mode
import urllib.request
import csv
import os

datasets_folder='raw_datasets/'
unsplash_folder='unsplash-research-dataset-lite-latest'
metObjects_folder='MetObjects'

# first let's load something from unsplash
file='photos.tsv000'
filepath=os.path.join(datasets_folder,unsplash_folder,file)

# load the file as a list of dicts
photos_list=[]
with open(filepath, 'r') as c:
    r = csv.DictReader(c,delimiter='\t')
    photos_list.extend(list(r))

# take a look at the data
for key, value in photos_list[0].items():
    print(f"{key}: {value}")

photo_id: bygTaBey1Xk
photo_url: https://unsplash.com/photos/bygTaBey1Xk
photo_image_url: https://images.unsplash.com/uploads/1413387620228d142bee4/23eceb86
photo_submitted_at: 2014-10-15 15:40:40.111061
photo_featured: t
photo_width: 4635
photo_height: 3070
photo_aspect_ratio: 1.51
photo_description: 
photographer_username: jaspervandermeij
photographer_first_name: Jasper
photographer_last_name: van der Meij
exif_camera_make: PENTAX RICOH IMAGING
exif_camera_model: GR
exif_iso: 100
exif_aperture_value: 14.0
exif_focal_length: 18.3
exif_exposure_time: 8
photo_location_name: 
photo_location_latitude: 
photo_location_longitude: 
photo_location_country: 
photo_location_city: 
stats_views: 1708356
stats_downloads: 19085
ai_description: sea and rock cliff with grasses under cloudy sky
ai_primary_landmark_name: Neist Point
ai_primary_landmark_latitude: 57.428386927437906
ai_primary_landmark_longitude: -6.7830279999999998
ai_primary_landmark_confidence: 30.348905999999999
blur_hash: LcE{wnIVR

In [4]:
# set this again just in case we run out of order
file='photos.tsv000'
filepath=os.path.join(datasets_folder,unsplash_folder,file)

# lets load it with only the data we want (the image url), and using the photo_id as a key
photos_dict={}
with open(filepath, 'r') as c:
    dictreader = csv.DictReader(c,delimiter='\t')
    for row in dictreader:
        photos_dict[row["photo_id"]]={"photo_image_url":row["photo_image_url"]}

print(f"Loaded {len(photos_dict)} photos, example item: {photos_dict['bygTaBey1Xk']}\n")

# now load the keywords and add the bits we want (keyword with highest confidence) to the photos
file='keywords.tsv000'
filepath=os.path.join(datasets_folder,unsplash_folder,file)
with open(filepath, 'r') as c:
    dictreader = csv.DictReader(c,delimiter='\t')
    for row in dictreader:
        key=row["photo_id"]
        keyword=row["keyword"]
        confidence=0 if row["ai_service_1_confidence"]=='' else float(row["ai_service_1_confidence"])
        if confidence>=photos_dict[key].get("ai_service_1_confidence",0):
            photos_dict[key]["keyword"]=keyword
            photos_dict[key]["ai_service_1_confidence"]=confidence

# let's check a handful, we can cross reference with our file for verification
for i, (key, value) in enumerate(photos_dict.items()):
    print(f"Photo ID {key}: {value}")
    if i > 10:
        break

Loaded 25000 photos, example item: {'photo_image_url': 'https://images.unsplash.com/uploads/1413387620228d142bee4/23eceb86'}

Photo ID bygTaBey1Xk: {'photo_image_url': 'https://images.unsplash.com/uploads/1413387620228d142bee4/23eceb86', 'keyword': 'outdoors', 'ai_service_1_confidence': 98.9488754272461}
Photo ID gXSFnk2a9V4: {'photo_image_url': 'https://images.unsplash.com/reserve/jEs6K0y1SbK3DAvgrBe5_IMG_3410.jpg', 'keyword': 'promontory', 'ai_service_1_confidence': 71.0499420166016}
Photo ID grg6-DNJuaU: {'photo_image_url': 'https://images.unsplash.com/uploads/141219200475673afcb68/f5bd8360', 'keyword': 'nature', 'ai_service_1_confidence': 98.0868301391602}
Photo ID sO42hhChB1c: {'photo_image_url': 'https://images.unsplash.com/reserve/ijl3tATFRpCjKWXwUoBz_DSCF7168.jpg', 'keyword': 'nature', 'ai_service_1_confidence': 81.94677734375}
Photo ID tkk8_HakQ98: {'photo_image_url': 'https://images.unsplash.com/reserve/6vaWXsQuSWSgm5NEF2p9_WC4A4194.jpg', 'keyword': 'soil', 'ai_service_1_conf

In [3]:
# now we have our data loaded and linked figure out how many distinct categories we have, and how unique they are
# not going to filter out low confidence results as this might actually be desirable!
categories={}
for photo_id,data in photos_dict.items():
    keyword=data["keyword"]
    if keyword in categories:
        categories[keyword]["total"]+=1
        categories[keyword]["instances"].append(photo_id)
    else:
        categories[keyword] = {"total":1,"instances":[photo_id]}

# let's see if we can get some stats
# categories_sorted = sorted(categories.items(), key=lambda x:x[1]["total"])
list_cats=[i["total"] for i in categories.values()]
print(f"We have {len(categories)} different categories. Range: {min(list_cats)} - {max(list_cats)}, Mean: {mean(list_cats)}, Median: {median(list_cats)}, Mode: {mode(list_cats)}\n")

# now print a few to check we have everything OK
for i, (key, value) in enumerate(categories.items()):
    print(f"Category {key} had {value['total']} instance(s), first instance is: {photos_dict[value['instances'][0]]}")
    if i > 10:
        break

We have 795 different categories. Range: 1 - 4901, Mean: 31.446540880503143, Median: 2, Mode: 1

Category outdoors had 1579 instance(s), first instance is: {'photo_image_url': 'https://images.unsplash.com/uploads/1413387620228d142bee4/23eceb86', 'keyword': 'outdoors', 'ai_service_1_confidence': 98.9488754272461}
Category promontory had 147 instance(s), first instance is: {'photo_image_url': 'https://images.unsplash.com/reserve/jEs6K0y1SbK3DAvgrBe5_IMG_3410.jpg', 'keyword': 'promontory', 'ai_service_1_confidence': 71.0499420166016}
Category nature had 4901 instance(s), first instance is: {'photo_image_url': 'https://images.unsplash.com/uploads/141219200475673afcb68/f5bd8360', 'keyword': 'nature', 'ai_service_1_confidence': 98.0868301391602}
Category soil had 408 instance(s), first instance is: {'photo_image_url': 'https://images.unsplash.com/reserve/6vaWXsQuSWSgm5NEF2p9_WC4A4194.jpg', 'keyword': 'soil', 'ai_service_1_confidence': 97.5186920166016}
Category stream had 8 instance(s), firs

In [4]:
# now we have our dataset lets download some images
download_folder='images/'
# set how many images to download
num_to_download=5000
# maximum failure rate
max_failures=25
# and a few variables to keep track of successes and failures
loaded=0
failed=0

while loaded<num_to_download and failed<=max_failures and loaded+failed<=len(photos_dict):
    exhausted_categories=[]
    for key, value in categories.items():
        # set up the photo_id, download URL and filepath
        photo_id=value["instances"].pop()
        photo_url=photos_dict[photo_id]["photo_image_url"]
        filename=f"{key}_{photo_id}.jpg"
        filepath=os.path.join(datasets_folder,download_folder,filename)
        # download the file (if it doesn't already exist)
        if os.path.isfile(filepath):
            loaded+=1
        else:
            try:
                urllib.request.urlretrieve(photo_url, filepath)
                loaded+=1
            except Exception as e:
                print(f"Download failed for {photo_url} with the following exception: {e}")
                failed+=1
        # if we've exhausted this category delete it
        if not value["instances"]:
            exhausted_categories.append(key)
        # break early if we need to
        if loaded>=num_to_download or failed>max_failures or loaded+failed>len(photos_dict):
            break
    # remove categories we've exhausted
    for key in exhausted_categories:
        del categories[key]

print(f"\n{loaded} images downloaded and saved in {os.path.join(datasets_folder,download_folder)}, {failed} downloads failed and were skipped.")

Download failed for https://images.unsplash.com/unsplash-premium-photos-production/premium_photo-1676667573119-40081df5d920 with the following exception: HTTP Error 404: Not Found
Download failed for https://images.unsplash.com/unsplash-premium-photos-production/premium_photo-1695219820032-34cfa7950b09 with the following exception: HTTP Error 404: Not Found
Download failed for https://images.unsplash.com/unsplash-premium-photos-production/premium_photo-1695635230516-e69891d27488 with the following exception: HTTP Error 404: Not Found
Download failed for https://images.unsplash.com/unsplash-premium-photos-production/premium_photo-1700567963303-1b83673c52a4 with the following exception: HTTP Error 404: Not Found
Download failed for https://images.unsplash.com/unsplash-premium-photos-production/premium_photo-1700984292461-fa2d83c28c6b with the following exception: HTTP Error 404: Not Found
Download failed for https://images.unsplash.com/unsplash-premium-photos-production/premium_photo-167